# if the more we increase the n_estimator in bagging step, we get the better result?
hipotesis: yeah it will increase score for model because model can generalize more better than just train once

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_breast_cancer

import warnings
warnings.filterwarnings('ignore')




In [2]:
# Load data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)

# Checking the original number of variables
original_feature_count = X.shape[1]
print('Original feature count: {}'.format(original_feature_count))
X.head()

Original feature count: 30


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [3]:
X_cancer = data.data
y_cancer = data.target

# Split data into training and testing dataset
X_train, X_test, y_train, y_test = train_test_split(X_cancer, y_cancer, stratify=y_cancer, random_state=66)

# Setting knn models and bagging configuration
models = {
    'KNN': KNeighborsClassifier(),
    'KNN + bagging': BaggingClassifier(KNeighborsClassifier(), n_estimators=10, random_state=0)
}

# Model construction
scores = pd.DataFrame(columns=["train_score", "test_score"], index=models.keys())

for model_name, model in models.items():
    model.fit(X_train, y_train)
    scores.loc[model_name, "train_score"] = model.score(X_train, y_train)
    scores.loc[model_name, "test_score"]  = model.score(X_test, y_test)

scores

,train_score,test_score
KNN,0.948357,0.923077
KNN + bagging,0.950704,0.93007


In [4]:
X_cancer = data.data
y_cancer = data.target

# Split data into training and testing dataset
X_train, X_test, y_train, y_test = train_test_split(X_cancer, y_cancer, stratify=y_cancer, random_state=66)

# Setting knn models and bagging configuration
models = {
    'KNN': KNeighborsClassifier(),
    'KNN + bagging': BaggingClassifier(KNeighborsClassifier(), n_estimators=100, random_state=0)
}

# Model construction
scores = pd.DataFrame(columns=["train_score", "test_score"], index=models.keys())

for model_name, model in models.items():
    model.fit(X_train, y_train)
    scores.loc[model_name, "train_score"] = model.score(X_train, y_train)
    scores.loc[model_name, "test_score"]  = model.score(X_test, y_test)

scores

,train_score,test_score
KNN,0.948357,0.923077
KNN + bagging,0.950704,0.937063


In [9]:
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=0)

base_models = {
    'LogisticRegression': LogisticRegression(max_iter=10000),
    'KNN': KNeighborsClassifier(),
    'DecisionTree': DecisionTreeClassifier()
}

rows = []
for name, base in base_models.items():
    base.fit(X_tr, y_tr)
    base_score = base.score(X_te, y_te)
    bag = BaggingClassifier(n_estimators=100, random_state=0).fit(X_tr, y_tr)
    bag_score = bag.score(X_te, y_te)
    rows.append({
        'base model':   name,
        'no bagging':   base_score,
        'bagging x100': bag_score,
        'lift':         bag_score - base_score,
    })

pd.DataFrame(rows).set_index('base model').round(4)

# Most classifiers get a better score with Bagging(n_estimators=100).
# However, LogisticRegression — despite its name, it is a classifier — shows 0 lift,
# because it is already a low-variance model and bagging adds no benefit.

,no bagging,bagging x100,lift
base model,,,
LogisticRegression,0.9371,0.9371,0.000
KNN,0.9161,0.9371,0.021
DecisionTree,0.9161,0.9371,0.021
